# Agentic RAG Log Analysis - Demo Notebook

This notebook demonstrates the Agentic RAG framework for log analysis.

In [ ]:
import sys
sys.path.append('..')

from src.input_layer.log_ingestion import LogIngestion
from src.preprocessing.log_preprocessor import LogPreprocessor
from src.knowledge_base.knowledge_manager import KnowledgeBaseManager
from src.retrieval.retrieval_system import RetrievalSystem
from src.llm_engine.llm_interface import LLMEngine
from src.agentic_controller.agentic_rag import AgenticController
from src.output_layer.alert_generator import AlertGenerator

## 1. Initialize Components

In [ ]:
# Initialize all components
ingestion = LogIngestion()
preprocessor = LogPreprocessor()
knowledge_manager = KnowledgeBaseManager()
retrieval_system = RetrievalSystem(knowledge_manager)
llm_engine = LLMEngine()
agentic_controller = AgenticController(retrieval_system, llm_engine)
alert_generator = AlertGenerator()

## 2. Setup Knowledge Base

In [ ]:
# Populate knowledge base with default documents
knowledge_manager.populate_default_knowledge()

# Check statistics
stats = knowledge_manager.get_statistics()
print(f"Knowledge Base Statistics:")
print(f"  Total Documents: {stats['total_documents']}")
print(f"  Document Types: {stats['document_types']}")
print(f"  Index Size: {stats['index_size']}")

## 3. Ingest and Preprocess Logs

In [ ]:
# Ingest sample logs
log_file = "../data/datasets/sample/sample_logs.log"
raw_logs = ingestion.ingest_file(log_file)

print(f"Ingested {len(raw_logs)} log entries")
print(f"\nFirst log entry:")
print(f"  Content: {raw_logs[0].content}")
print(f"  Timestamp: {raw_logs[0].timestamp}")

In [ ]:
# Preprocess logs
parsed_logs = preprocessor.preprocess(raw_logs)

print(f"Parsed {len(parsed_logs)} log entries")
print(f"\nFirst parsed entry:")
print(f"  Template: {parsed_logs[0].template}")
print(f"  Severity: {parsed_logs[0].severity}")
print(f"  Component: {parsed_logs[0].component}")

## 4. Analyze Single Log with Agentic RAG

In [ ]:
# Select an error log
error_log = [log for log in parsed_logs if log.severity in ['ERROR', 'CRITICAL', 'FATAL']][0]

print(f"Analyzing log: {error_log.raw_content}")
print(f"Severity: {error_log.severity}")

In [ ]:
# Run agentic analysis
analysis_result = agentic_controller.analyze_log(error_log)

print(f"\nAnalysis Result:")
print(f"  Is Anomaly: {analysis_result.is_anomaly}")
print(f"  Severity: {analysis_result.severity}")
print(f"  Confidence: {analysis_result.confidence_score:.2%}")
print(f"  Reasoning Steps: {len(analysis_result.reasoning_chain)}")

In [ ]:
# Display reasoning chain
print("\nReasoning Chain:")
for step in analysis_result.reasoning_chain:
    print(f"\nStep {step.step_number}:")
    print(f"  Thought: {step.thought}")
    print(f"  Action: {step.action.value}")
    print(f"  Observation: {step.observation[:200]}...")
    print(f"  Confidence: {step.confidence:.2%}")

In [ ]:
# Display final analysis
print("\nFinal Analysis:")
print(analysis_result.final_analysis)

## 5. Generate Alert

In [ ]:
# Generate alert from analysis
alert = alert_generator.generate_alert(analysis_result)

print(alert.to_human_readable())

## 6. Batch Analysis

In [ ]:
# Analyze all logs
all_results = []

for i, log in enumerate(parsed_logs[:5]):  # Analyze first 5 for demo
    print(f"Analyzing log {i+1}/5...")
    result = agentic_controller.analyze_log(log)
    all_results.append(result)

print(f"\nCompleted analysis of {len(all_results)} logs")

In [ ]:
# Generate alerts
alerts = alert_generator.generate_alerts_batch(all_results)

print(f"Generated {len(alerts)} alerts")

# Display alert statistics
stats = alert_generator.get_statistics(alerts)
print(f"\nAlert Statistics:")
print(f"  Total Alerts: {stats['total_alerts']}")
print(f"  Severity Distribution: {stats['severity_distribution']}")
print(f"  Average Confidence: {stats['average_confidence']:.2%}")

## 7. Test Retrieval System

In [ ]:
# Test retrieval with a query
query = "connection timeout error"
results = retrieval_system.retrieve(query, top_k=3)

print(f"Retrieved {len(results)} documents for query: '{query}'\n")

for result in results:
    print(f"Document: {result.document.title}")
    print(f"  Type: {result.document.doc_type}")
    print(f"  Relevance: {result.relevance_score:.3f}")
    print(f"  Content: {result.document.content[:150]}...")
    print()